# Species Classifier Recovery for Phase 2
Frozen ResNet-50, original 14 classes and grouped split, 224 px, up to 12 epochs. This creates a **new checkpoint**; Phase-1 results remain unchanged.

GPU recommended. CPU works with longer training time. Saves to Drive after every completed epoch; rerun setup and training after interruption to resume.

## 1. Setup and Restore Frozen Metadata

In [ ]:
from google.colab import drive
from pathlib import Path
import subprocess, sys, json, hashlib, zipfile, urllib.request, time

def mount_drive_preserving_local_files(drive_module, mountpoint='/content/drive'):
    import os
    mountpoint = Path(mountpoint)
    try:
        drive_module.mount(str(mountpoint))
    except ValueError as error:
        if 'Mountpoint must not already contain files' not in str(error):
            raise
        # Refuse to move any directory that contains an active filesystem mount.
        mountinfo = Path('/proc/self/mountinfo').read_text().splitlines()
        mounts = [line.split()[4].replace('\\040', ' ').replace('\\134', '\\') for line in mountinfo]
        prefix = str(mountpoint.resolve())
        if os.path.ismount(mountpoint) or any(path == prefix or path.startswith(prefix+'/') for path in mounts):
            raise RuntimeError('An active filesystem mount was detected here; local-folder backup was not attempted.') from error
        backup = mountpoint.with_name(mountpoint.name+'_local_backup_'+str(time.time_ns()))
        mountpoint.rename(backup)
        print('Preserved pre-mount local files:', backup)
        drive_module.mount(str(mountpoint))
mount_drive_preserving_local_files(drive)
ROOT = Path(input('Project folder (Enter for default): ').strip() or '/content/drive/MyDrive/CameraTrap_Phase2')
OUT = ROOT/'phase2_results'
RECOVERY = OUT/'species_recovery_v1'
RECOVERY.mkdir(parents=True, exist_ok=True)
subprocess.run([sys.executable,'-m','pip','install','-q',
               'transformers==4.49.0','pandas','scipy','ultralytics==8.3.221'],check=True)
BASE = 'https://raw.githubusercontent.com/salehdev3/Capstone_5/561822859f5c345f1e69ad90f83e68da0e56c3f2/'
SCRIPT_SHA256 = '283bcf3eedf2e54f21a1d22cded0f117d99f97953ed5b6bfd239fd9b9b9fee1f'
METADATA_SHA256 = '4f14b43168be69b915c74436d6de31931f169c3800f08d7051d220f11a72a2aa'
sha256 = lambda p: hashlib.sha256(Path(p).read_bytes()).hexdigest()
SCRIPT = RECOVERY/'train_species_recovery.py'
if SCRIPT.exists():
    assert sha256(SCRIPT)==SCRIPT_SHA256, 'Existing training source differs; preserve it before retrying'
else:
    temporary=SCRIPT.with_suffix('.download')
    urllib.request.urlretrieve(BASE+'tools/train_species_recovery.py',temporary)
    assert sha256(temporary)==SCRIPT_SHA256
    temporary.replace(SCRIPT)
archive=RECOVERY/'phase1_training_metadata.zip'
if archive.exists():
    assert sha256(archive)==METADATA_SHA256
else:
    temporary=archive.with_suffix('.download')
    urllib.request.urlretrieve(BASE+'releases/species_recovery/phase1_training_metadata.zip',temporary)
    assert sha256(temporary)==METADATA_SHA256
    temporary.replace(archive)
source=ROOT/'phase1_source'
with zipfile.ZipFile(archive) as z:
    assert z.testzip() is None
    for entry in z.infolist():
        target=(source/entry.filename).resolve()
        assert target.is_relative_to(source.resolve()), 'Unsafe metadata archive path'
        if target.exists() and not entry.is_dir():
            assert target.is_file() and target.read_bytes()==z.read(entry.filename), f'Existing frozen metadata differs: {target}'
    source.mkdir(parents=True,exist_ok=True)
    z.extractall(source)
assert sha256(source/'data/cct20_team_v2/manifest.json')=='d3bfb6ba0c9d31412f9d603a6f63ae7c4890f458c5d2aa2a87d7653bb882bb53'
assert (ROOT/'images').is_dir(), 'Original images missing: '+str(ROOT/'images')
print('Ready. Original dataset split preserved; no training has run yet.')
print('Results:',RECOVERY)


## 2. Train and Calibrate
Select checkpoints on `val_select`; fit temperature on `val_calibrate`. Test images are not used here. Original images must exist in `CameraTrap_Phase2/images`. Weights and resumable checkpoints are saved automatically to Drive.

In [ ]:
assert sha256(SCRIPT)==SCRIPT_SHA256
subprocess.run([sys.executable,'-u',str(SCRIPT),'--root',str(ROOT),'--stage','train'],check=True)
print('Saved:',ROOT/'models/species_recovery_v1/best.pt')
print('Saved:',ROOT/'models/species_recovery_v1/policy.json')


## 3. Historical Test Evaluation
Evaluate the new fixed checkpoint and calibration policy on the original 508 known-class test crops. This is historical evaluation, not a new independent test. Preserve the new results separately from Phase 1.

In [ ]:
assert sha256(SCRIPT)==SCRIPT_SHA256
subprocess.run([sys.executable,'-u',str(SCRIPT),'--root',str(ROOT),'--stage','test'],check=True)
import pandas as pd
results=json.loads((RECOVERY/'historical_test/metrics.json').read_text())
display(pd.DataFrame([{k:results[k] for k in ['n','accuracy','macro_f1_all_training_classes','ece','top3_accuracy','independent_test']}]))
display(pd.DataFrame(results['per_class']))


## 4. Initialize Member 4
Loads the new classifier together with the existing M4 detector and Recovery v2 masks. No detector or segmentation retraining. After initialization, continue with the updated HTTP/demo integration workflow.

In [ ]:
SPECIES_PT = ROOT/'models/species_recovery_v1/best.pt'
SPECIES_POLICY = ROOT/'models/species_recovery_v1/policy.json'
assert SPECIES_PT.is_file() and SPECIES_POLICY.is_file()
loader=RECOVERY/'member4_integration_colab.py'
urllib.request.urlretrieve(
    'https://raw.githubusercontent.com/salehdev3/Capstone_5/a560b89/tools/member4_integration_colab.py',loader)
exec(compile(loader.read_text(),str(loader),'exec'),globals())


## 5. Local HTTP Service and Recorded Demo
Five fixed validation cases: two previously detected annotated positives, the two earlier missed positives and one negative. Records actual outputs and verifies whether classification and masks were exercised. This is branch coverage, not an accuracy sample, benchmark or fallback video.

In [ ]:
from pathlib import Path
import subprocess,sys,json,hashlib,urllib.request,importlib.util

assert 'pipeline' in globals(), 'Initialize Member 4 integration in this same runtime first.'
ROOT=Path(globals().get('ROOT','/content/drive/MyDrive/CameraTrap_Phase2'))
OUT=Path(globals().get('OUT',ROOT/'phase2_results'))
subprocess.run([sys.executable,'-m','pip','install','-q','fastapi','uvicorn','python-multipart','requests'],check=True)
MODULE=OUT/'member4_recovery_v2/member4_service_v2.py'
MODULE.parent.mkdir(parents=True,exist_ok=True)
urllib.request.urlretrieve('https://raw.githubusercontent.com/salehdev3/Capstone_5/phase-2/tools/member4_service_v2.py',MODULE)
EXPECTED_SHA256='3c351637ba5a7d313628a3aba93816e5bebb345a8db00063652153f951e1bf54'
assert hashlib.sha256(MODULE.read_bytes()).hexdigest()==EXPECTED_SHA256
spec=importlib.util.spec_from_file_location('member4_service',MODULE)
module=importlib.util.module_from_spec(spec);spec.loader.exec_module(module)
# Stop only the previous service started by this cell, if it exists.
if '_member4_service_state' in globals():
    _member4_service_state['server'].should_exit=True
    _member4_service_state['thread'].join(timeout=5)
_member4_service_state=module.record_demo(pipeline,ROOT,OUT)
server=_member4_service_state['server']
thread=_member4_service_state['thread']
app=_member4_service_state['app']
SERVICE_URL=_member4_service_state['base_url']
demo_output=_member4_service_state['demo_output']
demo=_member4_service_state['demo']
IMAGES=_member4_service_state['images']
from IPython.display import display
from PIL import Image
import pandas as pd
if not demo_output['events']:print('No animal events were retained in this demo.')
else:display(pd.DataFrame(demo_output['events']))
for r in demo:
    im=Image.open(_member4_service_state['evidence_dir']/(r['id']+'_overlay.png'))
    im.thumbnail((640,480));display(im)
print('Saved:',_member4_service_state['evidence_dir'])


## 6. Event Counts, Full-Pipeline Timing and Deployment Decision
Recomputes fixed-detector counts on supervised historical test frames. Complete event coverage is enforced over retained frames. Measures 20 full-pipeline validation replays after warm-up. Loads exact matching v2 mask metrics and the new classifier test metrics. Uses unchanged preset acceptance targets. NO-GO describes unmet deployment criteria, not a course-completion grade.

In [ ]:
from pathlib import Path
import hashlib,urllib.request,importlib.util
assert 'pipeline' in globals() and 'demo' in globals(), 'Initialize integration and run the updated HTTP demo in this same runtime first.'
ROOT=Path(globals().get('ROOT','/content/drive/MyDrive/CameraTrap_Phase2'))
OUT=Path(globals().get('OUT',ROOT/'phase2_results'))
MODULE=OUT/'member4_recovery_v2/member4_evaluation_v2.py'
MODULE.parent.mkdir(parents=True,exist_ok=True)
urllib.request.urlretrieve('https://raw.githubusercontent.com/salehdev3/Capstone_5/phase-2/tools/member4_evaluation_v2.py',MODULE)
assert hashlib.sha256(MODULE.read_bytes()).hexdigest()=='6838f2c5c495918127d05ddc56429a4054534133895dd5f4bbdf9e23d64cf1df'
spec=importlib.util.spec_from_file_location('member4_evaluation',MODULE)
module=importlib.util.module_from_spec(spec);spec.loader.exec_module(module)
EVALUATION_DIR,acceptance=module.evaluate(pipeline,ROOT,OUT,demo)
from IPython.display import display
import pandas as pd
display(pd.DataFrame(acceptance))
print('Upload decision.json, event_metrics.json, pipeline_benchmark.json and acceptance.csv from:',EVALUATION_DIR)


## 7. Evidence Audit and Checkpoint Package
Checks current model identities and measured evidence, computes accepted-policy mask AP when possible, and creates a separate version-bound failure-review template. Packages actual trained checkpoints and available evidence in Drive even if items are missing. Missing report/slides/video or human reviews remain explicitly incomplete. No training.

Expected human deliverables: `submission/Phase2_Report.pdf`, `submission/Final_Presentation.pdf`, `submission/Demo.mp4`. A file-presence check is not content approval. W&B upload is a separate optional command after local save.

In [ ]:
from pathlib import Path
import subprocess,sys,json,hashlib,urllib.request,importlib.util
assert 'pipeline' in globals(), 'Initialize Member 4 integration in this same runtime first.'
ROOT=Path(globals().get('ROOT','/content/drive/MyDrive/CameraTrap_Phase2'))
OUT=Path(globals().get('OUT',ROOT/'phase2_results'))
subprocess.run([sys.executable,'-m','pip','install','-q','pycocotools'],check=True)
MODULE=OUT/'member4_recovery_v2/member4_audit_v2.py'
MODULE.parent.mkdir(parents=True,exist_ok=True)
urllib.request.urlretrieve('https://raw.githubusercontent.com/salehdev3/Capstone_5/phase-2/tools/member4_audit_v2.py',MODULE)
assert hashlib.sha256(MODULE.read_bytes()).hexdigest()=='bffdf56b7f3a1b6714b26acdd7293641426f8cdb3a99a165fda4b014ca3731aa'
spec=importlib.util.spec_from_file_location('member4_audit',MODULE)
module=importlib.util.module_from_spec(spec);spec.loader.exec_module(module)
AUDIT_DIR,ARCHIVE,AUDIT_STATUS=module.audit(pipeline,ROOT,OUT)
from IPython.display import display
import pandas as pd
display(pd.read_csv(AUDIT_DIR/'completion.csv'))
print('LOCAL PACKAGE SAVED:',ARCHIVE)
print('Send completion.csv and audit_status.json from:',AUDIT_DIR)

def upload_phase2_evidence_to_wandb():
    # Explicit optional upload after the local ZIP is safely written.
    subprocess.run([sys.executable,'-m','pip','install','-q','wandb'],check=True)
    import wandb
    wandb.login()
    with wandb.init(project='camera-trap-phase2',job_type='evidence-backup',
                    config={'segmentation_policy_sha256':pipeline.policy_sha256,'audit_status':AUDIT_STATUS['status']},
                    settings=wandb.Settings(init_timeout=30)) as run:
        artifact=wandb.Artifact('phase2-recovery-evidence',type='evaluation',metadata={'scope':'Versioned checkpoint/evidence package; inspect audit for incomplete items and deployment NO-GO.'})
        artifact.add_file(str(ARCHIVE),name=ARCHIVE.name)
        logged=run.log_artifact(artifact);logged.wait()
        link=dict(url=run.url,status='uploaded',artifact_name=logged.name,artifact_digest=logged.digest,
                  local_package_sha256=module.digest(ARCHIVE))
    module.write(OUT/'member4_recovery_v2/wandb_link.json',link)
    print('W&B backup uploaded. Rerun the audit cell to include this link in the next package.')

print('For a separate W&B package backup, run: upload_phase2_evidence_to_wandb()')


## 8. Optional W&B Model Backup
Back up the actual checkpoint and matching policy as a model artifact. Drive already contains the weights. Sign in to your own W&B account to run this optional cell.

In [ ]:
subprocess.run([sys.executable,'-m','pip','install','-q','wandb'],check=True)
import wandb
wandb.login()
with wandb.init(project='camera-trap-phase2',job_type='species-recovery-backup',config={'version':'species_recovery_v1','original_phase1_checkpoint':False}) as run:
    artifact=wandb.Artifact('species-recovery-v1',type='model',metadata={'source':'Retrained Phase-2 deployment classifier; not original Phase-1 weights'})
    for name in ['best.pt','policy.json','summary.json','config.json','history.csv','checkpoint_reload.json']:
        artifact.add_file(str(RECOVERY/name),name=name)
    logged=run.log_artifact(artifact)
    logged.wait()
    print('Backed up actual new weights and policy:',run.url)
